In [ ]:

#!/usr/bin/env python3
"""


Exact reproducibility and verification code for
"An explicit integral projection in weight 24 and a 2-adic Hecke
discrepancy"

What this script verifies
1) The 19x19 truncation matrix M for Trunc_18 on the frozen basis has det(M) != 0
2) The unique extraction coefficients beta_0..beta_18 exist for [q^2] pi(F)
3) The minimal 2-power clearing denominators is 2^30
4) gamma_n = 2^30 beta_n lies in Z_(2) for all n and v2(gamma_1) = 3
5) The recorded gamma_n mod 2^40 match exactly the appendix list
6) The V_24 basis has all positive coefficients divisible by 8 and its q^2 coefficients divisible by 32
7) For f0 = E4^3 and every odd prime p <= 499, S_p(f0) is divisible by 2^33

Dependencies
- Python 3.9+ recommended
- sympy

Run
hecke_discrepancy_verify.ipynb

Optional flags
--maxprime 499
--modH 40
"""

import math
import argparse
from typing import List, Tuple
import sympy as sp


def sigma_k(n: int, k: int) -> int:
    s = 0
    r = int(math.isqrt(n))
    for d in range(1, r + 1):
        if n % d == 0:
            s += d ** k
            e = n // d
            if e != d:
                s += e ** k
    return s


def eisenstein_E2(N: int) -> List[int]:
    a = [0] * (N + 1)
    a[0] = 1
    for n in range(1, N + 1):
        a[n] = -24 * sigma_k(n, 1)
    return a


def eisenstein_E4(N: int) -> List[int]:
    a = [0] * (N + 1)
    a[0] = 1
    for n in range(1, N + 1):
        a[n] = 240 * sigma_k(n, 3)
    return a


def eisenstein_E6(N: int) -> List[int]:
    a = [0] * (N + 1)
    a[0] = 1
    for n in range(1, N + 1):
        a[n] = -504 * sigma_k(n, 5)
    return a


def series_sub(a: List[int], b: List[int], N: int) -> List[int]:
    return [(a[i] if i < len(a) else 0) - (b[i] if i < len(b) else 0) for i in range(N + 1)]


def series_mul(a: List[int], b: List[int], N: int) -> List[int]:
    res = [0] * (N + 1)
    la, lb = len(a), len(b)
    for i in range(min(la, N + 1)):
        ai = a[i]
        if ai == 0:
            continue
        maxj = min(lb - 1, N - i)
        for j in range(maxj + 1):
            bj = b[j]
            if bj:
                res[i + j] += ai * bj
    return res


def series_pow(a: List[int], e: int, N: int) -> List[int]:
    res = [0] * (N + 1)
    res[0] = 1
    base = a[:]
    while e > 0:
        if e & 1:
            res = series_mul(res, base, N)
        e >>= 1
        if e:
            base = series_mul(base, base, N)
    return res


def monomial_series(E2: List[int], E4: List[int], E6: List[int], a: int, b: int, c: int, N: int) -> List[int]:
    s = [0] * (N + 1)
    s[0] = 1
    if a:
        s = series_mul(s, series_pow(E2, a, N), N)
    if b:
        s = series_mul(s, series_pow(E4, b, N), N)
    if c:
        s = series_mul(s, series_pow(E6, c, N), N)
    return s


def primes_upto(n: int) -> List[int]:
    sieve = [True] * (n + 1)
    sieve[0] = sieve[1] = False
    for p in range(2, int(n ** 0.5) + 1):
        if sieve[p]:
            step = p
            start = p * p
            sieve[start : n + 1 : step] = [False] * (((n - start) // step) + 1)
    return [i for i in range(2, n + 1) if sieve[i]]


def v2_int(n: int) -> int:
    n = abs(int(n))
    if n == 0:
        return 10**9
    v = 0
    while n % 2 == 0:
        n //= 2
        v += 1
    return v


def v2_rational(q: sp.Rational) -> int:
    q = sp.Rational(q)
    return v2_int(int(q.p)) - v2_int(int(q.q))


def mod_2H(q: sp.Rational, H: int) -> int:
    q = sp.Rational(q)
    mod = 1 << H
    num = int(q.p) % mod
    den = int(q.q) % mod
    if den % 2 == 0:
        raise ValueError("Denominator is even, not in Z_(2)")
    inv = pow(den, -1, mod)
    return (num * inv) % mod


def dot_mod(a: List[int], b: List[int], mod: int) -> int:
    s = 0
    for x, y in zip(a, b):
        s = (s + (x % mod) * (y % mod)) % mod
    return s


def hecke_coeffs(a: List[int], p: int, k: int, nmax: int) -> List[int]:
    b = [0] * (nmax + 1)
    pk = pow(p, k - 1)
    for n in range(nmax + 1):
        val = a[p * n] if p * n < len(a) else 0
        if n % p == 0:
            val += pk * (a[n // p] if (n // p) < len(a) else 0)
        b[n] = val
    return b


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--maxprime", type=int, default=499)
    ap.add_argument("--modH", type=int, default=40)
    args, _ = ap.parse_known_args()


    Ntr = 18
    maxprime = args.maxprime
    H = args.modH

    Nmax = Ntr * maxprime

    E2 = eisenstein_E2(Nmax)
    E4 = eisenstein_E4(Nmax)
    E6 = eisenstein_E6(Nmax)

    E4_3 = series_pow(E4, 3, Nmax)
    E6_2 = series_pow(E6, 2, Nmax)
    num = series_sub(E4_3, E6_2, Nmax)
    for x in num:
        if x % 1728 != 0:
            raise RuntimeError("Delta numerator not divisible by 1728")
    Delta = [x // 1728 for x in num]

    # Frozen bases from Appendix
    B12 = [
        (6, 0, 0),
        (4, 1, 0),
        (2, 2, 0),
        (0, 3, 0),
        (3, 0, 1),
        (1, 1, 1),
        (0, 0, 2),
    ]
    W24 = [(a, b + 3, c) for (a, b, c) in B12]
    V24 = [
        (0, 0, 4),
        (1, 1, 3),
        (2, 2, 2),
        (3, 0, 3),
        (4, 1, 2),
        (5, 2, 1),
        (6, 0, 2),
        (7, 1, 1),
        (8, 2, 0),
        (9, 0, 1),
        (10, 1, 0),
        (12, 0, 0),
    ]

    W24_series = [monomial_series(E2, E4, E6, a, b, c, Ntr) for (a, b, c) in W24]
    V24_series = [monomial_series(E2, E4, E6, a, b, c, Ntr) for (a, b, c) in V24]
    B24_series = W24_series + V24_series

    # Lemma V24div checks for the frozen basis
    for idx, s in enumerate(V24_series):
        for n in range(1, Ntr + 1):
            if s[n] % 8 != 0:
                raise RuntimeError(f"V24 basis element {idx} fails 8 divisibility at q^{n}")
        if s[2] % 32 != 0:
            raise RuntimeError(f"V24 basis element {idx} fails 32 divisibility at q^2")

    # Build truncation matrix M
    M = sp.Matrix([[B24_series[j][n] for j in range(19)] for n in range(19)])
    detM = sp.Integer(M.det())
    if detM == 0:
        raise RuntimeError("det(M) == 0, Trunc_18 not an isomorphism")
    print("OK det(M) != 0")

    # Projection pi is identity on first 7 basis elements and zero on last 12
    t = sp.Matrix([B24_series[j][2] if j < 7 else 0 for j in range(19)])

    # Solve M^T beta = t
    beta = M.T.LUsolve(t)
    v2_beta = [v2_rational(beta[i]) for i in range(19)]
    min_v2_beta = min(v2_beta)
    if min_v2_beta != -30:
        raise RuntimeError(f"min v2(beta_n) is {min_v2_beta}, expected -30")
    print("OK min v2(beta_n) = -30 so 2^30 is minimal")

    gamma = [sp.Rational(beta[i]) * (1 << 30) for i in range(19)]
    for i, g in enumerate(gamma):
        if int(g.q) % 2 == 0:
            raise RuntimeError(f"gamma_{i} has even denominator, not in Z_(2)")

    if v2_rational(gamma[1]) != 3:
        raise RuntimeError(f"v2(gamma_1) is {v2_rational(gamma[1])}, expected 3")
    print("OK v2(gamma_1) = 3")

    gamma_mod = [mod_2H(g, H) for g in gamma]
    print(f"Computed gamma_n mod 2^{H}")
    print(gamma_mod)

    appendix_gamma_mod40 = [
        949930897408,
        448494006680,
        768700897104,
        427689717038,
        286661271808,
        502521534252,
        180029225968,
        740716868213,
        900067543808,
        1026689374344,
        316908805344,
        500040199206,
        297024212288,
        613744187236,
        43675424040,
        737843413001,
        620949292416,
        602891394048,
        9518217136,
    ]

    if H == 40:
        if gamma_mod != appendix_gamma_mod40:
            raise RuntimeError("gamma mod 2^40 does not match appendix list")
        print("OK gamma_n mod 2^40 matches appendix list exactly")

    # Sanity check f0 = E4^3 is congruent to 1 mod 16 in positive degrees
    f0 = E4_3
    for n in range(1, Nmax + 1):
        if f0[n] % 16 != 0:
            raise RuntimeError(f"E4^3 fails mod 16 at q^{n}")
    print("OK E4^3 is 1 mod 16 in positive degrees")

    # Verify S_p(E4^3) divisible by 2^33 for odd primes up to maxprime
    Delta_f0 = series_mul(Delta, f0, Nmax)
    MODH = 1 << H
    MOD33 = 1 << 33

    gamma_modH = [mod_2H(g, H) for g in gamma]

    primes = [p for p in primes_upto(maxprime) if p % 2 == 1]
    bad = []
    for p in primes:
        Tp_Delta_f0 = hecke_coeffs(Delta_f0, p, 24, 18)
        term1 = dot_mod(gamma_modH, Tp_Delta_f0, MODH)

        Tp_f0 = hecke_coeffs(f0, p, 12, 2)
        Bp = Delta[0] * Tp_f0[2] + Delta[1] * Tp_f0[1] + Delta[2] * Tp_f0[0]
        term2 = ((1 << 30) * (Bp % MODH)) % MODH

        Sp_modH = (term1 - term2) % MODH
        if Sp_modH % MOD33 != 0:
            bad.append((p, int(Sp_modH % MOD33)))

    if bad:
        print("FAIL some primes did not satisfy divisibility by 2^33")
        print(bad[:10])
        raise SystemExit(1)

    print(f"OK S_p(E4^3) divisible by 2^33 for every odd prime p <= {maxprime}")

    print("All checks passed")


if __name__ == "__main__":
    main()
